In [ ]:
import boto3
import sagemaker
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
import os
import json

# Setup SageMaker session
session = sagemaker.Session()
region = 'us-east-1'
bucket = 'mva-python-code'  # your existing S3 bucket
role = 'arn:aws:iam::806575638863:role/SageMakerRole'

print("SageMaker SDK version:", sagemaker.__version__)
print("Region:", region)
print("Bucket:", bucket)

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /Users/madhavathavale/Library/Application Support/sagemaker/config.yaml
SageMaker SDK version: 2.257.3
Region: us-east-1
Bucket: mva-python-code


In [2]:
# Load iris dataset
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['target'] = iris.target

print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nTarget classes:", iris.target_names)
print("\nClass distribution:")
print(df['target'].value_counts())

Dataset shape: (150, 5)

First 5 rows:
   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                5.1               3.5                1.4               0.2   
1                4.9               3.0                1.4               0.2   
2                4.7               3.2                1.3               0.2   
3                4.6               3.1                1.5               0.2   
4                5.0               3.6                1.4               0.2   

   target  
0       0  
1       0  
2       0  
3       0  
4       0  

Target classes: ['setosa' 'versicolor' 'virginica']

Class distribution:
target
0    50
1    50
2    50
Name: count, dtype: int64


In [3]:
# Split into train and test
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

print(f"Train size: {len(train_df)}")
print(f"Test size: {len(test_df)}")

# Save locally first
os.makedirs('data', exist_ok=True)
train_df.to_csv('data/train.csv', index=False)
test_df.to_csv('data/test.csv', index=False)

# Upload to S3
s3_client = boto3.client('s3', region_name=region)

s3_client.upload_file('data/train.csv', bucket, 'iris/train/train.csv')
s3_client.upload_file('data/test.csv', bucket, 'iris/test/test.csv')

print("✓ Data uploaded to S3")
print(f"  s3://{bucket}/iris/train/train.csv")
print(f"  s3://{bucket}/iris/test/test.csv")

Train size: 120
Test size: 30
✓ Data uploaded to S3
  s3://mva-python-code/iris/train/train.csv
  s3://mva-python-code/iris/test/test.csv


In [4]:
# Write training script to disk
training_script = '''
import argparse
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib
import os

def model_fn(model_dir):
    model = joblib.load(os.path.join(model_dir, "model.joblib"))
    return model

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--n-estimators", type=int, default=100)
    parser.add_argument("--max-depth", type=int, default=5)
    parser.add_argument("--model-dir", type=str, default=os.environ.get("SM_MODEL_DIR"))
    parser.add_argument("--train", type=str, default=os.environ.get("SM_CHANNEL_TRAIN"))
    parser.add_argument("--test", type=str, default=os.environ.get("SM_CHANNEL_TEST"))
    args = parser.parse_args()

    # Load data
    train_df = pd.read_csv(os.path.join(args.train, "train.csv"))
    test_df = pd.read_csv(os.path.join(args.test, "test.csv"))

    # Split features and target
    X_train = train_df.drop("target", axis=1)
    y_train = train_df["target"]
    X_test = test_df.drop("target", axis=1)
    y_test = test_df["target"]

    # Train model
    print(f"Training RandomForest with {args.n_estimators} estimators...")
    model = RandomForestClassifier(
        n_estimators=args.n_estimators,
        max_depth=args.max_depth,
        random_state=42
    )
    model.fit(X_train, y_train)

    # Evaluate
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    print(f"Accuracy: {accuracy:.4f}")
    print("\\nClassification Report:")
    print(classification_report(y_test, predictions,
        target_names=["setosa", "versicolor", "virginica"]))

    # Save model
    joblib.dump(model, os.path.join(args.model_dir, "model.joblib"))
    print("Model saved!")
'''

os.makedirs('scripts', exist_ok=True)
with open('scripts/train.py', 'w') as f:
    f.write(training_script)

print("✓ Training script created")

✓ Training script created


In [5]:
from sagemaker.sklearn.estimator import SKLearn

# Create SKLearn estimator
sklearn_estimator = SKLearn(
    entry_point='train.py',
    source_dir='scripts',
    role=role,
    instance_type='ml.m5.large',
    framework_version='1.2-1',
    py_version='py3',
    hyperparameters={
        'n-estimators': 100,
        'max-depth': 5
    },
    output_path=f's3://{bucket}/iris/output',
    sagemaker_session=session
)

# Start training job
print("Starting training job...")
sklearn_estimator.fit({
    'train': f's3://{bucket}/iris/train',
    'test': f's3://{bucket}/iris/test'
})

print("✓ Training complete!")

Starting training job...


INFO:sagemaker:Creating training-job with name: sagemaker-scikit-learn-2026-05-21-00-23-18-598
ERROR:sagemaker:Please check the troubleshooting guide for common errors: https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-python-sdk-troubleshooting.html#sagemaker-python-sdk-troubleshooting-create-training-job


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:21                                                                                   │
│                                                                                                  │
│   18                                                                                             │
│   19 # Start training job                                                                        │
│   20 print("Starting training job...")                                                           │
│ ❱ 21 sklearn_estimator.fit({                                                                     │
│   22 │   'train': f's3://{bucket}/iris/train',                                                   │
│   23 │   'test': f's3://{bucket}/iris/test'                                                      │
│   24 })                                                                                          │
│                                                                                                  │
│ /opt/anaconda3/envs/bedrock-env/lib/python3.13/site-packages/sagemaker/telemetry/telemetry_loggi │
│ ng.py:171 in wrapper                                                                             │
│                                                                                                  │
│   168 │   │   │   │   │   caught_ex = e                                                          │
│   169 │   │   │   │   finally:                                                                   │
│   170 │   │   │   │   │   if caught_ex:                                                          │
│ ❱ 171 │   │   │   │   │   │   raise caught_ex                                                    │
│   172 │   │   │   │   │   return response  # pylint: disable=W0150                               │
│   173 │   │   │   else:                                                                          │
│   174 │   │   │   │   logger.debug(                                                              │
│                                                                                                  │
│ /opt/anaconda3/envs/bedrock-env/lib/python3.13/site-packages/sagemaker/telemetry/telemetry_loggi │
│ ng.py:142 in wrapper                                                                             │
│                                                                                                  │
│   139 │   │   │   │   start_timer = perf_counter()                                               │
│   140 │   │   │   │   try:                                                                       │
│   141 │   │   │   │   │   # Call the original function                                           │
│ ❱ 142 │   │   │   │   │   response = func(*args, **kwargs)                                       │
│   143 │   │   │   │   │   stop_timer = perf_counter()                                            │
│   144 │   │   │   │   │   elapsed = stop_timer - start_timer                                     │
│   145 │   │   │   │   │   extra += f"&x-latency={round(elapsed, 2)}"                             │
│                                                                                                  │
│ /opt/anaconda3/envs/bedrock-env/lib/python3.13/site-packages/sagemaker/workflow/pipeline_context │
│ .py:346 in wrapper                                                                               │
│                                                                                                  │
│   343 │   │   │                                                                                  │
│   344 │   │   │   return _StepArguments(retrieve_caller_name(self_instance), run_func, *args,    │
│   345 │   │                                                                                      │
│ ❱ 346 │   │   return run_func(*args, **kwargs)             